## Data Ingestion and Merging

In [1]:
import duckdb
import pandas as pd

In [2]:
# 1) Connect to our local database
con = duckdb.connect('../Processed_data/cyclistic_local_database.db')

In [3]:
# 2) Define the SQL query to  merge all 12 CSV files into one permanent table
merge_query = """
CREATE TABLE IF Not EXISTS raw_trips_data AS 
SELECT * FROM read_csv_auto('../Raw_Data/*.csv'); """

In [4]:
# 3) Execute the query using our active database connection
con.execute(merge_query)

In [5]:
# 4) Get the structural metadata of the merged table
schema_info = con.execute("PRAGMA table_info('raw_trips_data');").df()

In [6]:
# 5) Display only the cloumn names and their detected data types
schema_info[['name', 'type']]

,name,type
0,ride_id,VARCHAR
1,rideable_type,VARCHAR
2,started_at,TIMESTAMP
3,ended_at,TIMESTAMP
4,start_station_name,VARCHAR
5,start_station_id,VARCHAR
6,end_station_name,VARCHAR
7,end_station_id,VARCHAR
8,start_lat,DOUBLE
9,start_lng,DOUBLE


In [7]:
# 6) Execute a SQL query to count the total rows rows in our merged table
volume_result = con.execute("SELECT COUNT(*) AS total_raw_rows FROM raw_trips_data;").df()

In [8]:
# 7) Display the result in a clean format
volume_result

,total_raw_rows
0,5848703


In [9]:
# 8) Preview the first 5 rows of the merged table safely using LIMIT
preview_result = con.execute("SELECT * FROM raw_trips_data LIMIT 5;").df()

In [10]:
# 9) Display the preview dataframe
preview_result

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,FD0EB1D32AF0D47E,classic_bike,2026-01-31 09:13:09.018,2026-01-31 09:28:10.302,Central St & Girard Ave,CHI02042,Dodge Ave & Church St,CHI00741,42.064313,-87.686152,42.048308,-87.698224,member
1,FB27405C3F8C824F,classic_bike,2026-01-15 14:25:42.526,2026-01-15 14:33:18.854,Shore Dr & 55th St,CHI00394,Woodlawn Ave & 55th St,CHI00423,41.795212,-87.580715,41.795264,-87.596471,casual
2,6FAFA1709403AA27,electric_bike,2026-01-06 12:55:33.572,2026-01-06 13:02:17.922,Hampden Ct & Diversey Pkwy,CHI02087,None,None,41.932470,-87.642420,41.940000,-87.640000,member
3,1F34C1FAD9FEC2D8,electric_bike,2026-01-26 16:22:25.011,2026-01-26 16:53:15.197,Carpenter St & Huron St,CHI00286,None,None,41.894532,-87.653412,41.830000,-87.670000,member
4,8E3E3072D8D3D918,electric_bike,2026-01-10 18:13:30.139,2026-01-10 19:31:56.971,Clinton St & Madison St,CHI00233,None,None,41.881660,-87.641150,41.890000,-87.630000,member


In [11]:
# 10) Audit missing (Null or None) values across critical columns
quality_audit = con.execute("""
    SELECT
        COUNT(*) - COUNT(ride_id) AS null_ride_id,
        COUNT(*) - COUNT(started_at) AS null_started_at,
        COUNT(*) - COUNT(start_station_name) AS null_start_station_name,
        COUNT(*) - COUNT(end_station_name) AS null_end_station_name,
        COUNT(*) - COUNT(end_lat) AS null_end_lat,
    FROM raw_trips_data;        
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
quality_audit

,null_ride_id,null_started_at,null_start_station_name,null_end_station_name,null_end_lat
0,0,0,1249667,1314334,5896


In [13]:
# 11) Investigation missing start station names by grouping them by bike type
investigation_result = con.execute("""
    SELECT
        rideable_type,
        COUNT(*) AS total_null_rows
    FROM raw_trips_data
    WHERE start_station_name IS Null
    GROUP BY rideable_type;
""").df()

In [14]:
investigation_result

,rideable_type,total_null_rows
0,electric_bike,1249667


## Data Cleaning and Process

In [15]:
# 12) Transform and clean the data into a new permanent table
transformation_query = """
CREATE TABLE IF NOT EXISTS cleaned_trips_data AS
SELECT
    ride_id,
    rideable_type,
    started_at,
    ended_at,
    COALESCE(start_station_name, 'On-Street Parking') AS start_station_name,
    COALESCE(start_station_id, 'STREET') AS start_station_id,
    COALESCE(end_station_name, 'On-Street Parking') AS end_station_name,
    COALESCE(end_station_id, 'STREET') AS end_station_id,
    start_lat,
    start_lng,
    end_lat,
    end_lng,
    member_casual
FROM raw_trips_data
WHERE end_lat IS NOT NULL;
"""

In [16]:
# 13) Execute the transformation query
con.execute(transformation_query)

In [17]:
# 14) Count the total rows in the cleaned table to verify the filtering process
cleaned_volume = con.execute("SELECT COUNT(*) AS total_cleaned_rows FROM cleaned_trips_data;").df()

In [18]:
cleaned_volume

,total_cleaned_rows
0,5842807


In [19]:
# 15) Feature Engineering: Calculate ride lenght, day of week, and filter anomalies
feature_engineering_query = """
CREATE TABLE IF NOT EXISTS final_trips_data AS
SELECT
    ride_id,
    rideable_type,
    started_at,
    ended_at,
    date_diff('minute', started_at, ended_at) AS ride_lenght,
    dayname(started_at) AS day_of_week,
    start_station_name,
    start_station_id,
    end_station_name,
    end_station_id,
    start_lat,
    start_lng,
    end_lat,
    end_lng,
    member_casual
FROM cleaned_trips_data
WHERE date_diff('minute', started_at, ended_at) >=1
"""

In [20]:
con.execute(feature_engineering_query)

## Statistical Analysis

In [21]:
# 16) Analyze Phase: Calculate core descriptive statistics grouped by user type
descriptive_stats = con.execute("""
    SELECT
        member_casual,
        COUNT(*) AS total_rides,
        ROUND(AVG(ride_lenght),2) AS avg_duration_minutes,
        MAX(ride_lenght) AS max_duration_minutes
    FROM final_trips_data
    GROUP BY member_casual;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [22]:
descriptive_stats

,member_casual,total_rides,avg_duration_minutes,max_duration_minutes
0,member,3696922,12.25,1500
1,casual,2045240,19.39,1500


In [23]:
# 17) Temporal Analysis: Count rides and average duration by day of the week
temporal_stats = con.execute("""
    SELECT
        member_casual,
        dayname(started_at) AS day_of_week,
        dayofweek(started_at) AS day_num,
        COUNT(*) AS total_rides,
        ROUND(AVG(ride_lenght),2) AS avg_duration_minutes
    FROM final_trips_data
    GROUP BY member_casual, dayname(started_at), dayofweek(started_at)
    ORDER BY member_casual, day_num;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [24]:
temporal_stats

,member_casual,day_of_week,day_num,total_rides,avg_duration_minutes
0,casual,Sunday,0,350371,22.48
1,casual,Monday,1,239054,19.53
2,casual,Tuesday,2,228953,16.90
3,casual,Wednesday,3,230601,16.14
4,casual,Thursday,4,260449,16.96
5,casual,Friday,5,314967,18.91
6,casual,Saturday,6,420845,21.74
7,member,Sunday,0,408688,13.43
8,member,Monday,1,520239,12.00
9,member,Tuesday,2,584685,11.81


In [25]:
# 18) Vehical Analysis: Count rides and average duration by bike type and user type
bike_type_stats = con.execute("""
    SELECT
        member_casual,
        rideable_type,
        COUNT(*) AS total_rides,
        ROUND(AVG(ride_lenght),2) AS avg_duration_ride
    FROM final_trips_data
    GROUP BY member_casual, rideable_type
    ORDER BY member_casual, total_rides;
""").df()

In [26]:
bike_type_stats

,member_casual,rideable_type,total_rides,avg_duration_ride
0,casual,classic_bike,657171,29.59
1,casual,electric_bike,1388069,14.57
2,member,classic_bike,1283112,14.04
3,member,electric_bike,2413810,11.29


In [27]:
# 19) Share Phase: Exprort aggregated dataframes to CSV for tableau visulization
descriptive_stats.to_csv('cyclistic_descriptive_stats.csv', index=False)
temporal_stats.to_csv('cyclistic_temporal_stats.csv', index=False)
bike_type_stats.to_csv('cyclistic_bike_type_stats.csv', index=False)